In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName('OlistData')\
        .getOrCreate()

In [0]:
#connect ADLS to Databricks
spark.conf.set("fs.azure.account.key.adlsgen2spark01.dfs.core.windows.net",
                 "Storageaccount -> Security + Networking -> Access Keys-->Key1valuePaste here ")

In [0]:
#basePath
adlsgen2CSVpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/"
adlsgen2PARQUETpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/parquet/"


In [0]:
#to view the datasets present isnide the container
display(dbutils.fs.ls(adlsgen2CSVpath))

path,name,size,modificationTime
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_customers_dataset/,olist_customers_dataset/,0,1778315425000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_geolocation_dataset/,olist_geolocation_dataset/,0,1778315428000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_order_items_dataset/,olist_order_items_dataset/,0,1778315432000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_order_payments_dataset/,olist_order_payments_dataset/,0,1778315434000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_order_reviews_dataset/,olist_order_reviews_dataset/,0,1778315436000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_orders_dataset/,olist_orders_dataset/,0,1778315439000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_products_dataset/,olist_products_dataset/,0,1778315440000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/olist_sellers_dataset/,olist_sellers_dataset/,0,1778315442000
abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/product_category_name_translation/,product_category_name_translation/,0,1778315443000


In [0]:
customers_df = spark.read.csv(adlsgen2CSVpath + "olist_customers_dataset", header=True, inferSchema=True) 
geolocation_df = spark.read.csv(adlsgen2CSVpath + "olist_geolocation_dataset", header=True, inferSchema=True) 
order_items_df = spark.read.csv(adlsgen2CSVpath + "olist_order_items_dataset", header=True, inferSchema=True) 
order_payments_df = spark.read.csv(adlsgen2CSVpath + "olist_order_payments_dataset", header=True, inferSchema=True) 
order_reviews_df = spark.read.csv(adlsgen2CSVpath + "olist_order_reviews_dataset", header=True, inferSchema=True) 
orders_df = spark.read.csv(adlsgen2CSVpath + "olist_orders_dataset", header=True, inferSchema=True) 
products_df = spark.read.csv(adlsgen2CSVpath + "olist_products_dataset", header=True, inferSchema=True) 
sellers_df = spark.read.csv(adlsgen2CSVpath + "olist_sellers_dataset", header=True, inferSchema=True) 
catgeory_translation_df = spark.read.csv(adlsgen2CSVpath + "product_category_name_translation", header=True, inferSchema=True) 

In [0]:
customers_df.show(5)
# customer_df.inferSchema()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


In [0]:
#Data Leakage or Drop - to compare while reading in adls gen2 did we not loose any data - so the count should match

print(f'Customers: {customers_df.count()} rows')
print(f'GeoLocation: {geolocation_df.count()} rows')
print(f'Order Items: {order_items_df.count()} rows')
print(f'Order Payments: {order_payments_df.count()} rows')
print(f'Order Reviews: {order_reviews_df.count()} rows')
print(f'Orders: {orders_df.count()} rows')
print(f'Products: {products_df.count()} rows')
print(f'Sellers: {sellers_df.count()} rows')
print(f'Category Translation: {catgeory_translation_df.count()} rows')


Customers: 99441 rows
GeoLocation: 1000163 rows
Order Items: 112650 rows
Order Payments: 103886 rows
Order Reviews: 104162 rows
Orders: 99441 rows
Products: 32951 rows
Sellers: 3095 rows
Category Translation: 71 rows


In [0]:
from pyspark.sql.functions import col, when, count

#check for nulls in critical fields
print(f'Customers: {customers_df.select(count(when(col('customer_id').isNull(), 'customer_id'))).collect()[0][0]} nulls') #for single col

#for all columns
customers_df.select([count(when(col(c).isNull(), c)).alias(c) for c in customers_df.columns]).show()


geolocation_df.select([count(when(col(c).isNull(),c)).alias(c) for c in geolocation_df.columns]).show()

order_items_df.select([count(when(col(c).isNull(),c)).alias(c) for c in order_items_df.columns]).show()

Customers: 0 nulls
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+

+--------+-------------+----------+---------+--------------

In [0]:
#Duplicate values

customers_df.groupBy('customer_id').count().filter(col('count')>1).show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
#customer distribution by State

customers_df.groupBy('customer_state').count().orderBy('count',ascending = False).show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows


In [0]:
# Order - Order status distribution

orders_df.groupBy('order_status').count().orderBy('count',ascending = False).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [0]:
#payments

order_payments_df.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [0]:
from pyspark.sql.functions import sum

top_products = order_items_df.groupBy('product_id').agg(sum('price').alias('total_sales'))
top_products.orderBy('total_sales',ascending = False).show(20)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Average Delivery Time analysis

delivery_df = orders_df.select('order_id','order_purchase_timestamp','order_delivered_customer_date')
delivery_df.show()

+--------------------+------------------------+-----------------------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+------------------------+-----------------------------+
|cd0d42e029e3b56ba...|     2018-02-28 16:27:24|          2018-04-07 14:24:39|
|85cecbd067fc6feb9...|     2017-08-18 22:30:29|          2017-08-29 20:23:06|
|70cca242632c1f58a...|     2018-04-06 21:36:54|          2018-04-13 15:36:46|
|d063dcc119860db74...|     2018-08-21 01:06:28|          2018-08-29 20:48:33|
|ba436db3c441f63a8...|     2018-07-31 19:00:14|          2018-08-03 17:22:54|
|eaa0211ae0667cb75...|     2017-08-21 12:27:20|          2017-08-24 20:33:41|
|82ffe097d8ddbf319...|     2018-04-23 01:41:29|          2018-04-24 17:34:30|
|885ab07cf96454b41...|     2018-01-11 19:35:01|          2018-01-31 00:48:51|
|be13c8dcfef552661...|     2018-04-20 10:21:38|          2018-04-30 19:24:42|
|1aabc8c7c8525b6ad...|     2018-04-23 17:32:07|          2018-04

In [0]:
from pyspark.sql.functions import datediff, to_date
delivery_detail_df = delivery_df.withColumn('delivery_time', datediff(col("order_delivered_customer_date"),col("order_purchase_timestamp"))).orderBy("delivery_time",ascending = False).show()

+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|ca07593549f1816d2...|     2017-02-21 23:31:27|          2017-09-19 14:36:39|          210|
|1b3190b2dfa9d789e...|     2018-02-23 14:57:35|          2018-09-19 23:24:07|          208|
|440d0d17af552815d...|     2017-03-07 23:59:51|          2017-09-19 15:12:50|          196|
|2fb597c2f772eca01...|     2017-03-08 18:09:02|          2017-09-19 14:33:17|          195|
|285ab9426d6982034...|     2017-03-08 22:47:40|          2017-09-19 14:00:04|          195|
|0f4519c5f1c541dde...|     2017-03-09 13:26:57|          2017-09-19 14:38:21|          194|
|47b40429ed8cce3ae...|     2018-01-03 09:44:01|          2018-07-13 20:51:31|          191|
|2fe324febf907e3ea...|     2017-03-13 20:17:10|          2017-09-19 17:00:07|   